In [7]:
import geopandas as gpd
from shapely.geometry import LineString

def countries_near_route(coordinates, buffer_km=50, country_shapefile="ne_10m_admin_0_countries.shp"):
    """
    Find countries near a given polyline route.

    Parameters:
        polyline: list of (lon, lat) tuples representing the route
        buffer_km: distance in kilometers to consider "near" the route
        country_shapefile: path to country polygons (Natural Earth recommended)

    Returns:
        List of country names near the route
    """
    # Load country polygons
    countries = gpd.read_file(country_shapefile)

    # Convert polyline to LineString
    polyline = [(c.latitude, c.longitude) for c in coordinates]
    line = LineString(polyline)

    # Reproject to a metric CRS for accurate buffering
    countries = countries.to_crs(epsg=3857)  # meters
    line = gpd.GeoSeries([line], crs="EPSG:4326").to_crs(epsg=3857).iloc[0]

    # Create buffer around route
    buffer = line.buffer(buffer_km * 1000)  # km → meters

    # Spatial index query for efficiency
    possible_matches_index = list(countries.sindex.intersection(buffer.bounds))
    possible_matches = countries.iloc[possible_matches_index]

    # Precise intersection check
    intersecting_countries = possible_matches[possible_matches.intersects(buffer)]

    return intersecting_countries['NAME'].tolist()




In [5]:
import datetime

from app.data.dto.main.Coordinates import Coordinates
from app.services.db_service import DbService
from app.services.external_api.searoute_api import SearouteApi

from dotenv import load_dotenv
load_dotenv()

sql_db_service = DbService()
await sql_db_service.init_pool()
searoute_api = SearouteApi("https://api.searoutes.com/", "EWhTo2x2hihNDCPZjCaMgFDWGegJoVLnYP7mqi5L")

In [9]:
route, err = await sql_db_service.get_route_by_id("f5a102f0-5769-4e85-a947-fdb0199a67e8")
departure_port, err = await sql_db_service.get_port_by_locode("RULED")
destination_port, err = await sql_db_service.get_port_by_locode("AEJEA")
sea_route, err = await searoute_api.build_sea_route(departure_port.latitude, departure_port.longitude, destination_port.latitude, destination_port.longitude, is_plan=True, speed_in_knots=10, departure_dt=datetime.datetime.now())

In [10]:
# Example usage
countries = countries_near_route(sea_route.seaRouteCoordinates, country_shapefile="./data/ne_10m_admin_0_countries.shp")
print(countries)


['Italy', 'San Marino', 'Mozambique', 'Tanzania', 'Kenya', 'Ethiopia', 'Eritrea', 'Yemen', 'Oman', 'Sudan', 'Bir Tawil', 'Egypt', 'Iran', 'Greece', 'Poland', 'Belarus', 'Lithuania', 'Latvia', 'Estonia', 'Finland', 'Austria', 'Czechia', 'Germany', 'Sweden']


In [14]:
import geopandas as gpd

countries = gpd.read_file("./data/ne_10m_admin_0_countries.shp")

# Example: ISO alpha-2 list
iso2_list = sorted(countries["ISO_A2"].unique())

# Example: English short names
name_list = sorted(countries["NAME"].unique())

print(iso2_list)
print(name_list)

['-99', 'AD', 'AE', 'AF', 'AG', 'AI', 'AL', 'AM', 'AO', 'AQ', 'AR', 'AS', 'AT', 'AU', 'AW', 'AX', 'AZ', 'BA', 'BB', 'BD', 'BE', 'BF', 'BG', 'BH', 'BI', 'BJ', 'BL', 'BM', 'BN', 'BO', 'BR', 'BS', 'BT', 'BW', 'BY', 'BZ', 'CA', 'CD', 'CF', 'CG', 'CH', 'CI', 'CK', 'CL', 'CM', 'CN', 'CN-TW', 'CO', 'CR', 'CU', 'CV', 'CW', 'CY', 'CZ', 'DE', 'DJ', 'DK', 'DM', 'DO', 'DZ', 'EC', 'EE', 'EG', 'EH', 'ER', 'ES', 'ET', 'FI', 'FJ', 'FK', 'FM', 'FO', 'GA', 'GB', 'GD', 'GE', 'GG', 'GH', 'GI', 'GL', 'GM', 'GN', 'GQ', 'GR', 'GS', 'GT', 'GU', 'GW', 'GY', 'HK', 'HM', 'HN', 'HR', 'HT', 'HU', 'ID', 'IE', 'IL', 'IM', 'IN', 'IO', 'IQ', 'IR', 'IS', 'IT', 'JE', 'JM', 'JO', 'JP', 'KE', 'KG', 'KH', 'KI', 'KM', 'KN', 'KP', 'KR', 'KW', 'KY', 'KZ', 'LA', 'LB', 'LC', 'LI', 'LK', 'LR', 'LS', 'LT', 'LU', 'LV', 'LY', 'MA', 'MC', 'MD', 'ME', 'MF', 'MG', 'MH', 'MK', 'ML', 'MM', 'MN', 'MO', 'MP', 'MR', 'MS', 'MT', 'MU', 'MV', 'MW', 'MX', 'MY', 'MZ', 'NA', 'NC', 'NE', 'NF', 'NG', 'NI', 'NL', 'NP', 'NR', 'NU', 'NZ', 'OM', 'PA',

In [1]:
import time
import asyncpg
from app.data.dto.main.SeaPort import SeaPortDB

db_name="BunkeringBot"
db_user="def_user"
db_password="super_password"
db_host="77.37.96.222"
db_port="5432"

connection_pool = await asyncpg.create_pool(
            database=db_name,
            user=db_user,
            password=db_password,
            host=db_host,
            port=db_port,
            min_size=1,
            max_size=20
        )

In [3]:
ports = []

async with connection_pool.acquire() as conn:
    rows = await conn.fetch(
        """
        SELECT *
        FROM public.ports_vector_new
        WHERE port_name = 'Ningbo'
        """
    )

    for r in rows:
        p = SeaPortDB.from_db_row(r)
        ports.append(p)


In [4]:
ports

[SeaPortDB(locode='CNNGB', country_code='', country_name='China', port_name='Ningbo', latitude=29.936067, longitude=121.844334, rank_score=None, similarity_score=None, combined_score=None, match_type='unknown', mabux_ids=[351, 115, 644], port_size='large', mabux_id=351, barge_status=None, truck_status=None, agent_contact_list=None, manual_input=False, bubble_id='1713117929766x113157264185515410', search_key='ningbo|cnngb|china|351', id='9d7eeb19-6a95-4c58-87d8-7cd33f1d9003'),
 SeaPortDB(locode='CNNBO', country_code='', country_name='China', port_name='Ningbo', latitude=29.866667, longitude=121.55, rank_score=None, similarity_score=None, combined_score=None, match_type='unknown', mabux_ids=[351, 115, 644], port_size='tiny', mabux_id=None, barge_status=None, truck_status=None, agent_contact_list=None, manual_input=False, bubble_id='1713117929769x224342594977439140', search_key='ningbo|cnnbo|china|', id='270be2bc-0edf-46c4-93d1-759a0574b85a')]

In [32]:
db_country_codes = set([p.locode[:2] for p in ports])

In [34]:
set(iso2_list) ^ db_country_codes

{'-99',
 'AD',
 'AF',
 'AN',
 'AX',
 'BF',
 'BI',
 'BN',
 'BQ',
 'BT',
 'BZ',
 'CF',
 'CN-TW',
 'ET',
 'FM',
 'FR',
 'GF',
 'GG',
 'GL',
 'GP',
 'GU',
 'HN',
 'IM',
 'IO',
 'JE',
 'KE',
 'KG',
 'KY',
 'LA',
 'LI',
 'LS',
 'LU',
 'MC',
 'ME',
 'MF',
 'MH',
 'ML',
 'MN',
 'MP',
 'MQ',
 'MW',
 'NE',
 'NF',
 'NI',
 'NO',
 'NR',
 'PS',
 'PW',
 'PY',
 'RE',
 'RW',
 'SJ',
 'SS',
 'SV',
 'TC',
 'TD',
 'TJ',
 'TK',
 'TW',
 'UG',
 'UZ',
 'XZ',
 'YT',
 'ZM'}